<a href="https://colab.research.google.com/github/paul-niloy-3407/Comparative-Analysis-with-Diabetes-Prediction-Dataset-ML-/blob/main/Comparative_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install imbalanced-learn --quiet
!pip install xgboost --quiet

In [2]:
from google.colab import drive

In [3]:
import os

In [ ]:
drive.mount('/content/drive')

In [5]:
# ── Google Drive Paths ─────────────────────────────────────
DRIVE_BASE    = '/content/drive/MyDrive/ICDAI - 2026/'
DATASET_PATH  = DRIVE_BASE + 'archive/diabetes_prediction_dataset.csv'
RESULTS_PATH  = DRIVE_BASE + 'Results/'

In [6]:
os.makedirs(RESULTS_PATH, exist_ok=True)

In [ ]:
print("Google Drive mounted successfully.")
print(f"Dataset will load from : {DATASET_PATH}")
print(f"All results will save to: {RESULTS_PATH}")

In [8]:
import pandas            as pd
import numpy             as np
import warnings
warnings.filterwarnings('ignore')

In [9]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn           as sns

In [10]:
plt.rcParams['figure.dpi']      = 120
plt.rcParams['font.size']       = 11
plt.rcParams['axes.titlesize']  = 13
plt.rcParams['axes.titleweight']= 'bold'
plt.rcParams['axes.labelsize']  = 11
sns.set_style('whitegrid')
PALETTE = ['#534AB7', '#1D9E75', '#EF9F27', '#E24B4A',
           '#0C447C', '#854F0B', '#712B13']

In [11]:
from sklearn.preprocessing     import StandardScaler, LabelEncoder
from sklearn.model_selection   import (train_test_split,
                                        GridSearchCV,
                                        RandomizedSearchCV,
                                        StratifiedKFold,
                                        cross_val_score)
from sklearn.feature_selection import SelectKBest, f_classif

In [12]:
from sklearn.linear_model  import LogisticRegression
from sklearn.tree          import DecisionTreeClassifier
from sklearn.ensemble      import (RandomForestClassifier,
                                   VotingClassifier)
from sklearn.svm           import SVC
from sklearn.neighbors     import KNeighborsClassifier
from sklearn.naive_bayes   import GaussianNB
from xgboost               import XGBClassifier

In [13]:
from sklearn.metrics import (accuracy_score,
                              precision_score,
                              recall_score,
                              f1_score,
                              roc_auc_score,
                              roc_curve,
                              confusion_matrix,
                              classification_report,
                              ConfusionMatrixDisplay)

In [14]:
from imblearn.over_sampling import SMOTE
from scipy.stats import randint, uniform

In [ ]:
diabetes_df = pd.read_csv(DATASET_PATH)

print(f"Dataset loaded successfully!")
print(f"Total rows    : {diabetes_df.shape[0]:,}")
print(f"Total columns : {diabetes_df.shape[1]}")
print(f"\nColumn names  : {list(diabetes_df.columns)}")

In [ ]:
print("\n── First 5 rows ──")
print(diabetes_df.head())

print("\n── Data types of each column ──")
print(diabetes_df.dtypes)

print("\n── Basic statistics (numerical columns) ──")
print(diabetes_df.describe().round(2))

print("\n── Missing values per column ──")
missing_values = diabetes_df.isnull().sum()
print(missing_values)
print(f"\nTotal missing values: {missing_values.sum()}")

print("\n── Target column distribution ──")
target_counts = diabetes_df['diabetes'].value_counts()
print(target_counts)
diabetic_pct     = (target_counts[1] / len(diabetes_df)) * 100
non_diabetic_pct = (target_counts[0] / len(diabetes_df)) * 100
print(f"\nDiabetic     : {target_counts[1]:,} ({diabetic_pct:.1f}%)")
print(f"Non-Diabetic : {target_counts[0]:,} ({non_diabetic_pct:.1f}%)")
print(f"\nClass imbalance ratio: 1 : {target_counts[0]/target_counts[1]:.1f}")

In [ ]:
numerical_columns   = ['age', 'bmi', 'HbA1c_level', 'blood_glucose_level']
binary_columns      = ['hypertension', 'heart_disease']
categorical_columns = ['gender', 'smoking_history']
target_column       = 'diabetes'

print("Numerical columns   :", numerical_columns)
print("Binary columns      :", binary_columns)
print("Categorical columns :", categorical_columns)
print("Target column       :", target_column)

# Unique values for categorical columns
for col in categorical_columns:
    print(f"\n{col} unique values: {diabetes_df[col].unique()}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Figure 1 — Univariate Analysis: Distribution of Numerical Features',
             fontsize=14, fontweight='bold', y=1.01)

histogram_colors = ['#534AB7', '#1D9E75', '#EF9F27', '#E24B4A']

for index, (column_name, color) in enumerate(
        zip(numerical_columns, histogram_colors)):
    row    = index // 2
    column = index  % 2
    ax     = axes[row][column]

    ax.hist(diabetes_df[column_name].dropna(),
            bins=40, color=color, edgecolor='white',
            linewidth=0.5, alpha=0.85)

    ax.set_title(f'Distribution of {column_name}')
    ax.set_xlabel(column_name)
    ax.set_ylabel('Number of Patients')
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

    mean_val   = diabetes_df[column_name].mean()
    median_val = diabetes_df[column_name].median()
    ax.axvline(mean_val,   color='black',  linestyle='--',
               linewidth=1.5, label=f'Mean: {mean_val:.2f}')
    ax.axvline(median_val, color='gray',   linestyle=':',
               linewidth=1.5, label=f'Median: {median_val:.2f}')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(RESULTS_PATH + 'fig01_histograms_numerical.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig01_histograms_numerical.png")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 6))
fig.suptitle('Figure 2 — Univariate Analysis: Box Plots of Numerical Features'
             '\n(Showing median, IQR, and outliers)',
             fontsize=13, fontweight='bold')

for index, (column_name, color) in enumerate(
        zip(numerical_columns, histogram_colors)):
    ax = axes[index]

    bp = ax.boxplot(diabetes_df[column_name].dropna(),
                    patch_artist=True, notch=False,
                    widths=0.5,
                    boxprops=dict(facecolor=color, alpha=0.7),
                    medianprops=dict(color='black', linewidth=2),
                    whiskerprops=dict(linewidth=1.5),
                    capprops=dict(linewidth=1.5),
                    flierprops=dict(marker='o', markersize=2,
                                    alpha=0.3, color=color))

    q1  = diabetes_df[column_name].quantile(0.25)
    q3  = diabetes_df[column_name].quantile(0.75)
    iqr = q3 - q1
    outliers = ((diabetes_df[column_name] < q1 - 1.5 * iqr) |
                (diabetes_df[column_name] > q3 + 1.5 * iqr)).sum()

    ax.set_title(f'{column_name}')
    ax.set_ylabel('Value')
    ax.set_xticks([])
    ax.text(0.5, 0.01, f'Outliers: {outliers:,}',
            transform=ax.transAxes, ha='center',
            fontsize=9, color='gray')

plt.tight_layout()
plt.savefig(RESULTS_PATH + 'fig02_boxplots_numerical.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig02_boxplots_numerical.png")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Figure 3 — Univariate Analysis: Density (KDE) Plots of Numerical Features',
             fontsize=13, fontweight='bold')

for index, (column_name, color) in enumerate(
        zip(numerical_columns, histogram_colors)):
    row    = index // 2
    column = index  % 2
    ax     = axes[row][column]

    diabetes_df[column_name].plot.kde(ax=ax, color=color, linewidth=2.5)
    ax.fill_between(
        ax.lines[0].get_xdata(),
        ax.lines[0].get_ydata(),
        alpha=0.25, color=color)

    skewness = diabetes_df[column_name].skew()
    ax.set_title(f'Density of {column_name}\n(Skewness: {skewness:.3f})')
    ax.set_xlabel(column_name)
    ax.set_ylabel('Density')

plt.tight_layout()
plt.savefig(RESULTS_PATH + 'fig03_kde_density_plots.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig03_kde_density_plots.png")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Figure 4 — Univariate Analysis: Target Class Distribution (Diabetes)',
             fontsize=13, fontweight='bold')

class_labels  = ['Non-Diabetic (0)', 'Diabetic (1)']
class_counts  = [target_counts[0], target_counts[1]]
bar_colors    = ['#1D9E75', '#E24B4A']

In [22]:
bars = axes[0].bar(class_labels, class_counts,
                   color=bar_colors, edgecolor='white',
                   linewidth=1.2, width=0.5)
for bar, count in zip(bars, class_counts):
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 500,
                 f'{count:,}\n({count/len(diabetes_df)*100:.1f}%)',
                 ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[0].set_title('Class Distribution — Bar Chart')
axes[0].set_xlabel('Diabetes Status')
axes[0].set_ylabel('Number of Patients')
axes[0].set_ylim(0, max(class_counts) * 1.15)
axes[0].yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

In [ ]:
wedges, texts, autotexts = axes[1].pie(
    class_counts,
    labels=class_labels,
    colors=bar_colors,
    autopct='%1.1f%%',
    startangle=140,
    wedgeprops=dict(edgecolor='white', linewidth=2),
    textprops=dict(fontsize=11))
for autotext in autotexts:
    autotext.set_fontweight('bold')
axes[1].set_title('Class Distribution — Pie Chart')

plt.tight_layout()
plt.savefig(RESULTS_PATH + 'fig04_target_class_distribution.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig04_target_class_distribution.png")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle('Figure 5 — Univariate Analysis: Categorical Feature Distributions',
             fontsize=13, fontweight='bold')

categorical_palette = ['#534AB7', '#1D9E75', '#EF9F27',
                       '#E24B4A', '#0C447C', '#854F0B']

# Gender — Bar chart
gender_counts = diabetes_df['gender'].value_counts()
axes[0][0].bar(gender_counts.index, gender_counts.values,
               color=categorical_palette[:len(gender_counts)],
               edgecolor='white', linewidth=1)
for i, (label, count) in enumerate(gender_counts.items()):
    axes[0][0].text(i, count + 200, f'{count:,}',
                    ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0][0].set_title('Gender Distribution')
axes[0][0].set_xlabel('Gender')
axes[0][0].set_ylabel('Number of Patients')
axes[0][0].yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

axes[0][1].pie(gender_counts.values,
               labels=gender_counts.index,
               colors=categorical_palette[:len(gender_counts)],
               autopct='%1.1f%%',
               startangle=90,
               wedgeprops=dict(edgecolor='white', linewidth=2))
axes[0][1].set_title('Gender — Proportion')

# Smoking History — Bar chart
smoking_counts = diabetes_df['smoking_history'].value_counts()
axes[1][0].barh(smoking_counts.index, smoking_counts.values,
                color=categorical_palette[:len(smoking_counts)],
                edgecolor='white', linewidth=1)
for i, (label, count) in enumerate(smoking_counts.items()):
    axes[1][0].text(count + 200, i, f'{count:,}',
                    va='center', fontsize=9)
axes[1][0].set_title('Smoking History Distribution')
axes[1][0].set_xlabel('Number of Patients')
axes[1][0].set_ylabel('Smoking History Category')
axes[1][0].xaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Smoking History — Pie chart
axes[1][1].pie(smoking_counts.values,
               labels=smoking_counts.index,
               colors=categorical_palette[:len(smoking_counts)],
               autopct='%1.1f%%',
               startangle=140,
               wedgeprops=dict(edgecolor='white', linewidth=1.5),
               textprops=dict(fontsize=9))
axes[1][1].set_title('Smoking History — Proportion')

plt.tight_layout()
plt.savefig(RESULTS_PATH + 'fig05_categorical_distributions.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig05_categorical_distributions.png")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Figure 6 — Univariate Analysis: Binary Feature Distributions',
             fontsize=13, fontweight='bold')

binary_col_colors = [['#1D9E75', '#E24B4A'], ['#534AB7', '#EF9F27']]
binary_col_labels = [['No Hypertension', 'Has Hypertension'],
                     ['No Heart Disease', 'Has Heart Disease']]

for idx, (col_name, colors, labels) in enumerate(
        zip(binary_columns, binary_col_colors, binary_col_labels)):
    counts = diabetes_df[col_name].value_counts().sort_index()
    bars   = axes[idx].bar(labels, counts.values,
                           color=colors, edgecolor='white',
                           linewidth=1.2, width=0.5)
    for bar, count in zip(bars, counts.values):
        axes[idx].text(bar.get_x() + bar.get_width() / 2,
                       bar.get_height() + 300,
                       f'{count:,}\n({count/len(diabetes_df)*100:.1f}%)',
                       ha='center', va='bottom',
                       fontsize=10, fontweight='bold')
    axes[idx].set_title(f'{col_name.replace("_", " ").title()} Distribution')
    axes[idx].set_xlabel('Status')
    axes[idx].set_ylabel('Number of Patients')
    axes[idx].set_ylim(0, max(counts.values) * 1.18)
    axes[idx].yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig(RESULTS_PATH + 'fig06_binary_feature_distributions.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig06_binary_feature_distributions.png")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 6))
fig.suptitle('Figure 7 — Bivariate Analysis: Numerical Features vs Diabetes Status',
             fontsize=13, fontweight='bold')

target_palette = {0: '#1D9E75', 1: '#E24B4A'}

for idx, col_name in enumerate(numerical_columns):
    ax = axes[idx]

    group_0 = diabetes_df[diabetes_df['diabetes'] == 0][col_name]
    group_1 = diabetes_df[diabetes_df['diabetes'] == 1][col_name]

    bp = ax.boxplot([group_0, group_1],
                    patch_artist=True,
                    labels=['Non-Diabetic\n(0)', 'Diabetic\n(1)'],
                    widths=0.5,
                    boxprops=dict(alpha=0.8),
                    medianprops=dict(color='black', linewidth=2.5),
                    flierprops=dict(marker='o', markersize=1, alpha=0.2))

    box_colors = ['#1D9E75', '#E24B4A']
    for patch, color in zip(bp['boxes'], box_colors):
        patch.set_facecolor(color)

    ax.set_title(col_name)
    ax.set_xlabel('Diabetes Status')
    ax.set_ylabel(col_name)

    mean_0 = group_0.mean()
    mean_1 = group_1.mean()
    ax.text(0.5, 0.97,
            f'Mean (0): {mean_0:.1f}\nMean (1): {mean_1:.1f}',
            transform=ax.transAxes, ha='center', va='top',
            fontsize=8.5, color='black',
            bbox=dict(boxstyle='round,pad=0.3',
                      facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig(RESULTS_PATH + 'fig07_bivariate_boxplots_vs_target.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig07_bivariate_boxplots_vs_target.png")

In [ ]:
df_for_corr = diabetes_df.copy()
for col in categorical_columns:
    df_for_corr[col] = LabelEncoder().fit_transform(
        df_for_corr[col].astype(str))

correlation_matrix = df_for_corr.corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))

heatmap = sns.heatmap(
    correlation_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    annot_kws={'size': 9},
    cbar_kws={'label': 'Correlation Coefficient', 'shrink': 0.8},
    ax=ax)

ax.set_title('Figure 8 — Bivariate Analysis: Pearson Correlation Heatmap\n'
             '(All Features including Encoded Categoricals)',
             fontsize=13, fontweight='bold', pad=15)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=10)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10)

plt.tight_layout()
plt.savefig(RESULTS_PATH + 'fig08_correlation_heatmap.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig08_correlation_heatmap.png")

In [ ]:
scatter_pairs = [
    ('HbA1c_level',         'blood_glucose_level'),
    ('age',                 'bmi'),
    ('age',                 'blood_glucose_level'),
    ('bmi',                 'HbA1c_level'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle('Figure 9 — Bivariate Analysis: Scatter Plots of Key Feature Pairs\n'
             '(Colored by Diabetes Status)',
             fontsize=13, fontweight='bold')

scatter_colors = {0: '#1D9E75', 1: '#E24B4A'}
scatter_labels = {0: 'Non-Diabetic', 1: 'Diabetic'}
scatter_sample = diabetes_df.sample(n=5000, random_state=42)

for idx, (x_col, y_col) in enumerate(scatter_pairs):
    row    = idx // 2
    column = idx  % 2
    ax     = axes[row][column]

    for class_value in [0, 1]:
        subset = scatter_sample[scatter_sample['diabetes'] == class_value]
        ax.scatter(subset[x_col], subset[y_col],
                   c=scatter_colors[class_value],
                   label=scatter_labels[class_value],
                   alpha=0.45, s=18, edgecolors='none')

    ax.set_xlabel(x_col, fontsize=10)
    ax.set_ylabel(y_col, fontsize=10)
    ax.set_title(f'{x_col}  vs  {y_col}')
    ax.legend(fontsize=9, markerscale=1.5)

plt.tight_layout()
plt.savefig(RESULTS_PATH + 'fig09_scatter_plots.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig09_scatter_plots.png")

In [ ]:
scatter_matrix_cols = numerical_columns + ['diabetes']
scatter_matrix_sample = diabetes_df[scatter_matrix_cols].sample(
    n=3000, random_state=42)

pair_plot = sns.pairplot(
    scatter_matrix_sample,
    hue='diabetes',
    palette={0: '#1D9E75', 1: '#E24B4A'},
    plot_kws={'alpha': 0.4, 's': 12},
    diag_kind='kde',
    corner=False)

pair_plot.fig.suptitle(
    'Figure 10 — Bivariate Analysis: Scatter Plot Matrix (Pair Plot)\n'
    'Green = Non-Diabetic  |  Red = Diabetic  '
    '(3,000 random samples)',
    y=1.02, fontsize=12, fontweight='bold')

pair_plot.savefig(RESULTS_PATH + 'fig10_scatter_plot_matrix.png',
                  dpi=120, bbox_inches='tight')
plt.show()
print("Saved: fig10_scatter_plot_matrix.png")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Figure 11 — Bivariate Analysis: Categorical Features vs Diabetes Status',
             fontsize=13, fontweight='bold')

for idx, col_name in enumerate(categorical_columns):
    cross_tab = pd.crosstab(diabetes_df[col_name],
                            diabetes_df['diabetes'],
                            normalize='index') * 100

    cross_tab.plot(kind='bar',
                   ax=axes[idx],
                   color=['#1D9E75', '#E24B4A'],
                   edgecolor='white',
                   linewidth=0.8,
                   width=0.6)

    axes[idx].set_title(f'{col_name.replace("_", " ").title()} vs Diabetes Rate (%)')
    axes[idx].set_xlabel(col_name.replace('_', ' ').title())
    axes[idx].set_ylabel('Percentage of Patients (%)')
    axes[idx].legend(['Non-Diabetic (0)', 'Diabetic (1)'], fontsize=9)
    axes[idx].tick_params(axis='x', rotation=25)

    for container in axes[idx].containers:
        axes[idx].bar_label(container, fmt='%.1f%%',
                             fontsize=8, padding=2)

plt.tight_layout()
plt.savefig(RESULTS_PATH + 'fig11_categorical_vs_target.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig11_categorical_vs_target.png")

In [ ]:
preprocessed_df = diabetes_df.copy()
gender_encoder  = LabelEncoder()
preprocessed_df['gender'] = gender_encoder.fit_transform(
    preprocessed_df['gender'].astype(str))
print(f"gender encoded  : {dict(zip(gender_encoder.classes_,
                                    gender_encoder.transform(gender_encoder.classes_)))}")

In [ ]:
smoking_encoder = LabelEncoder()
preprocessed_df['smoking_history'] = smoking_encoder.fit_transform(
    preprocessed_df['smoking_history'].astype(str))
print(f"smoking encoded : {dict(zip(smoking_encoder.classes_,
                                    smoking_encoder.transform(smoking_encoder.classes_)))}")

# Separate features (X) and target (y)
X_features = preprocessed_df.drop(columns=['diabetes'])
y_target   = preprocessed_df['diabetes']

print(f"\nFeature matrix shape : {X_features.shape}")
print(f"Target vector shape  : {y_target.shape}")
print(f"Feature names        : {list(X_features.columns)}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_features,
    y_target,
    test_size    = 0.20,
    random_state = 42,
    stratify     = y_target   # preserve class ratio in both sets
)

print("Train-Test Split complete.")
print(f"Training set : {X_train.shape[0]:,} rows  ({X_train.shape[0]/len(X_features)*100:.0f}%)")
print(f"Test set     : {X_test.shape[0]:,} rows  ({X_test.shape[0]/len(X_features)*100:.0f}%)")
print(f"\nTraining class distribution:")
print(y_train.value_counts())
print(f"\nTest class distribution:")
print(y_test.value_counts())

In [ ]:
feature_scaler = StandardScaler()

# Fit only on training data → learn mean and std from training
X_train_scaled = feature_scaler.fit_transform(X_train)

# Apply SAME scaling to test data (do NOT re-fit)
X_test_scaled  = feature_scaler.transform(X_test)

print("Feature scaling complete.")
print(f"Training features  — mean ≈ {X_train_scaled.mean():.4f}, std ≈ {X_train_scaled.std():.4f}")
print(f"Test features      — mean ≈ {X_test_scaled.mean():.4f}, std ≈ {X_test_scaled.std():.4f}")

In [ ]:
print("Before SMOTE — Training class distribution:")
print(pd.Series(y_train).value_counts())

smote_balancer = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote_balancer.fit_resample(
    X_train_scaled, y_train)

print("\nAfter SMOTE — Training class distribution:")
smote_counts = pd.Series(y_train_smote).value_counts()
print(smote_counts)
print(f"\nTotal training samples after SMOTE : {len(X_train_smote):,}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Figure 12 — SMOTE: Class Distribution Before and After Balancing',
             fontsize=13, fontweight='bold')

before_counts = pd.Series(y_train).value_counts().sort_index()
after_counts  = pd.Series(y_train_smote).value_counts().sort_index()
class_labels  = ['Non-Diabetic (0)', 'Diabetic (1)']
bar_colors    = ['#1D9E75', '#E24B4A']

for ax, counts, title_suffix in zip(
        axes,
        [before_counts, after_counts],
        ['Before SMOTE (Imbalanced)', 'After SMOTE (Balanced)']):
    bars = ax.bar(class_labels, counts.values,
                  color=bar_colors, edgecolor='white',
                  linewidth=1.2, width=0.5)
    for bar, count in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 100,
                f'{count:,}',
                ha='center', va='bottom',
                fontsize=11, fontweight='bold')
    ax.set_title(title_suffix)
    ax.set_xlabel('Diabetes Status')
    ax.set_ylabel('Number of Samples')
    ax.set_ylim(0, max(counts.values) * 1.15)
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig(RESULTS_PATH + 'fig12_smote_before_after.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig12_smote_before_after.png")

In [ ]:
feature_selector = SelectKBest(score_func=f_classif, k='all')
feature_selector.fit(X_train_smote, y_train_smote)

feature_scores = pd.DataFrame({
    'Feature'   : X_features.columns,
    'F-Score'   : feature_selector.scores_,
    'P-Value'   : feature_selector.pvalues_
}).sort_values('F-Score', ascending=False).reset_index(drop=True)

print("Feature Importance Ranking (SelectKBest — ANOVA F-test):")
print(feature_scores.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

colors_by_score = [
    '#E24B4A' if score == feature_scores['F-Score'].max()
    else '#534AB7'
    for score in feature_scores['F-Score']
]

bars = ax.barh(feature_scores['Feature'],
               feature_scores['F-Score'],
               color=colors_by_score,
               edgecolor='white', linewidth=0.8)

for bar, score in zip(bars, feature_scores['F-Score']):
    ax.text(bar.get_width() + 50,
            bar.get_y() + bar.get_height() / 2,
            f'{score:,.0f}',
            va='center', fontsize=9)

ax.set_title('Figure 13 — Feature Importance: SelectKBest ANOVA F-Scores\n'
             '(Higher score = stronger relationship with diabetes diagnosis)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('ANOVA F-Score (Higher is More Important)')
ax.set_ylabel('Feature Name')
ax.invert_yaxis()

plt.tight_layout()
plt.savefig(RESULTS_PATH + 'fig13_feature_importance_selectkbest.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig13_feature_importance_selectkbest.png")

In [ ]:
rf_for_importance = RandomForestClassifier(
    n_estimators=100, random_state=42, n_jobs=-1)
rf_for_importance.fit(X_train_smote, y_train_smote)

rf_importance_df = pd.DataFrame({
    'Feature'    : X_features.columns,
    'Importance' : rf_for_importance.feature_importances_
}).sort_values('Importance', ascending=False).reset_index(drop=True)

print("\nRandom Forest Feature Importance:")
print(rf_importance_df.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

rf_colors = [
    '#E24B4A' if imp == rf_importance_df['Importance'].max()
    else '#1D9E75'
    for imp in rf_importance_df['Importance']
]

ax.barh(rf_importance_df['Feature'],
        rf_importance_df['Importance'],
        color=rf_colors,
        edgecolor='white', linewidth=0.8)

for idx, (_, row) in enumerate(rf_importance_df.iterrows()):
    ax.text(row['Importance'] + 0.002, idx,
            f"{row['Importance']:.4f}",
            va='center', fontsize=9)

ax.set_title('Figure 14 — Feature Importance: Random Forest Gini Impurity\n'
             '(Red bar = most important feature)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Feature Importance Score (Gini Impurity Reduction)')
ax.set_ylabel('Feature Name')
ax.invert_yaxis()

plt.tight_layout()
plt.savefig(RESULTS_PATH + 'fig14_feature_importance_randomforest.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig14_feature_importance_randomforest.png")

In [41]:
def evaluate_trained_model(model, X_test_data, y_test_data):
    """
    Given a trained classifier, compute 5 performance metrics on test data.
    Returns a dictionary with percentage values (rounded to 2 decimals).
    """
    y_predicted       = model.predict(X_test_data)
    y_probabilities   = model.predict_proba(X_test_data)[:, 1]

    return {
        'Accuracy (%)' : round(accuracy_score (y_test_data, y_predicted)                     * 100, 2),
        'Precision (%)': round(precision_score(y_test_data, y_predicted, zero_division=0)    * 100, 2),
        'Recall (%)'   : round(recall_score   (y_test_data, y_predicted)                     * 100, 2),
        'F1-Score (%)' : round(f1_score       (y_test_data, y_predicted)                     * 100, 2),
        'ROC-AUC (%)'  : round(roc_auc_score  (y_test_data, y_probabilities)                 * 100, 2),
    }

In [ ]:
MODEL_DISPLAY_NAMES = {
    'Logistic Regression' : 'LR',
    'Decision Tree'       : 'DT',
    'Random Forest'       : 'RF',
    'SVM'                 : 'SVM',
    'KNN'                 : 'KNN',
    'Naive Bayes'         : 'NB',
    'XGBoost'             : 'XGB',
}

print("Helper function defined. Ready to train models.")

In [ ]:
dhp_models_dict = {
    'Logistic Regression' : LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree'       : DecisionTreeClassifier(random_state=42),
    'Random Forest'       : RandomForestClassifier(random_state=42, n_jobs=-1),
    'SVM'                 : SVC(probability=True, random_state=42),
    'KNN'                 : KNeighborsClassifier(),
    'Naive Bayes'         : GaussianNB(),
    'XGBoost'             : XGBClassifier(random_state=42,
                                           eval_metric='logloss',
                                           verbosity=0),
}

dhp_results_dict = {}
dhp_trained_dict = {}

for model_name, model_object in dhp_models_dict.items():
    model_object.fit(X_train_smote, y_train_smote)
    metrics = evaluate_trained_model(
        model_object, X_test_scaled, y_test)
    dhp_results_dict[model_name] = metrics
    dhp_trained_dict[model_name] = model_object

    print(f"  {model_name:22s} | "
          f"Acc: {metrics['Accuracy (%)']:6.2f}% | "
          f"F1: {metrics['F1-Score (%)']:6.2f}% | "
          f"AUC: {metrics['ROC-AUC (%)']:6.2f}%")

print("\nDHP training complete.")

In [44]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

gscv_param_grids = {
    'Logistic Regression': {
        'C': [0.1, 1, 10],
        'solver': ['lbfgs', 'liblinear']
    },
    'Decision Tree': {
        'max_depth': [5, 10, None],
        'min_samples_split': [2, 10],
        'criterion': ['gini', 'entropy']
    },
    'Random Forest': {
        'n_estimators': [100, 200],
        'max_depth': [10, None]
    },
    'SVM': {
        'estimator__C': [0.1, 1, 10]
    },
    'KNN': {
        'n_neighbors': [5, 11, 15],
        'weights': ['uniform', 'distance']
    },
    'Naive Bayes': {
        'var_smoothing': [1e-9, 1e-8, 1e-7]
    },
    'XGBoost': {
        'n_estimators': [100, 200],
        'max_depth': [3, 6],
        'learning_rate': [0.1, 0.3]
    }
}

gscv_base_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42, n_jobs=-1),
    'SVM': CalibratedClassifierCV(estimator=LinearSVC(random_state=42, max_iter=5000)),
    'KNN': KNeighborsClassifier(),
    'Naive Bayes': GaussianNB(),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss', verbosity=0, n_jobs=-1)
}

gscv_results_dict = {}
gscv_trained_dict = {}
gscv_best_params_dict = {}

for model_name, base_model in gscv_base_models.items():
    print(f"\nSearching: {model_name} ...")
    grid_search = GridSearchCV(
        estimator=base_model,
        param_grid=gscv_param_grids[model_name],
        cv=3,
        scoring='f1',
        refit=True,
        n_jobs=-1,
        verbose=0,
        error_score='raise'
    )
    grid_search.fit(X_train_smote, y_train_smote)
    best_model = grid_search.best_estimator_
    metrics = evaluate_trained_model(best_model, X_test_scaled, y_test)
    gscv_results_dict[model_name] = metrics
    gscv_trained_dict[model_name] = best_model
    gscv_best_params_dict[model_name] = grid_search.best_params_
    print(f"Acc: {metrics['Accuracy (%)']:6.2f}% | F1: {metrics['F1-Score (%)']:6.2f}% | AUC:{metrics['ROC-AUC (%)']:6.2f}%")
    print("Best Params:", grid_search.best_params_)

print("\nGSCV complete ✓")

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

rscv_param_dists = {
    'Logistic Regression': {
        'C': uniform(0.01, 50),
        'solver': ['lbfgs', 'liblinear']
    },
    'Decision Tree': {
        'max_depth': [None, 5, 10, 20],
        'min_samples_split': randint(2, 15),
        'criterion': ['gini', 'entropy']
    },
    'Random Forest': {
        'n_estimators': randint(100, 300),
        'max_depth': [None, 10, 20],
        'min_samples_split': randint(2, 10)
    },
    'SVM': {
        'estimator__C': uniform(0.1, 20)
    },
    'KNN': {
        'n_neighbors': randint(5, 20),
        'weights': ['uniform', 'distance'],
        'metric': ['euclidean', 'manhattan']
    },
    'Naive Bayes': {
        'var_smoothing': uniform(1e-10, 1e-6)
    },
    'XGBoost': {
        'n_estimators': randint(100, 300),
        'max_depth': randint(3, 8),
        'learning_rate': uniform(0.05, 0.25),
        'subsample': uniform(0.7, 0.3)
    }
}

rscv_base_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42, n_jobs=-1),
    'SVM': CalibratedClassifierCV(estimator=LinearSVC(random_state=42, max_iter=5000)),
    'KNN': KNeighborsClassifier(),
    'Naive Bayes': GaussianNB(),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss', verbosity=0, n_jobs=-1)
}

rscv_results_dict = {}
rscv_trained_dict = {}
rscv_best_params_dict = {}

for model_name, base_model in rscv_base_models.items():
    print(f"\nSearching: {model_name} ...")
    random_search = RandomizedSearchCV(
        estimator=base_model,
        param_distributions=rscv_param_dists[model_name],
        n_iter=15,
        cv=3,
        scoring='f1',
        random_state=42,
        refit=True,
        n_jobs=-1,
        verbose=0,
        error_score='raise'
    )
    random_search.fit(X_train_smote, y_train_smote)
    best_model = random_search.best_estimator_
    metrics = evaluate_trained_model(best_model, X_test_scaled, y_test)
    rscv_results_dict[model_name] = metrics
    rscv_trained_dict[model_name] = best_model
    rscv_best_params_dict[model_name] = random_search.best_params_
    print(f"Acc: {metrics['Accuracy (%)']:6.2f}% | F1: {metrics['F1-Score (%)']:6.2f}% | AUC:{metrics['ROC-AUC (%)']:6.2f}%")
    print("Best Params:", random_search.best_params_)

print("\nRSCV complete ✓")

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

stratified_kfold = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

kfold_models_dict = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'SVM': CalibratedClassifierCV(estimator=LinearSVC(random_state=42, max_iter=5000)),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes': GaussianNB(),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss', verbosity=0, n_jobs=-1)
}

kfold_results_dict = {}

for model_name, model_object in kfold_models_dict.items():
    print(f"\nCV Running: {model_name} ...")
    # One pass computes both metrics
    cv_scores = cross_validate(
        estimator=model_object,
        X=X_train_smote,
        y=y_train_smote,
        cv=stratified_kfold,
        scoring={'accuracy':'accuracy', 'f1':'f1'},
        n_jobs=-1
    )

    kfold_results_dict[model_name] = {
        'Mean Accuracy (%)': round(cv_scores['test_accuracy'].mean()*100, 2),
        'Std Accuracy (%)': round(cv_scores['test_accuracy'].std()*100, 2),
        'Mean F1 (%)': round(cv_scores['test_f1'].mean()*100, 2),
        'Std F1 (%)': round(cv_scores['test_f1'].std()*100, 2)
    }

    r = kfold_results_dict[model_name]
    print(f"Accuracy: {r['Mean Accuracy (%)']:.2f}% ± {r['Std Accuracy (%)']:.2f}% | "
          f"F1: {r['Mean F1 (%)']:.2f}% ± {r['Std F1 (%)']:.2f}%")

print("\n5-Fold Cross Validation complete ✓")

In [ ]:

from sklearn.ensemble import VotingClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV


fast_svm = CalibratedClassifierCV(
estimator=LinearSVC(
random_state=42,
max_iter=5000
)
)


voting_classifier = VotingClassifier(

estimators=[

    (
        'logistic_regression',
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    ),

    (
        'random_forest',
        RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        )
    ),

    (
        'fast_svm',
        fast_svm
    ),

    (
        'xgboost',
        XGBClassifier(
            random_state=42,
            eval_metric='logloss',
            verbosity=0,
            n_jobs=-1
        )
    ),

    (
        'knn',
        KNeighborsClassifier(
            n_neighbors=5
        )
    )
],

# soft voting uses probabilities
voting='soft',

n_jobs=-1


)


print("\nTraining voting ensemble...")

voting_classifier.fit(
X_train_smote,
y_train_smote
)


voting_metrics = evaluate_trained_model(
voting_classifier,
X_test_scaled,
y_test
)

print("\nVoting Ensemble Results")
print("-" * 50)

print(f"Accuracy  : {voting_metrics['Accuracy (%)']:.2f}%")
print(f"Precision : {voting_metrics['Precision (%)']:.2f}%")
print(f"Recall    : {voting_metrics['Recall (%)']:.2f}%")
print(f"F1-Score  : {voting_metrics['F1-Score (%)']:.2f}%")
print(f"ROC-AUC   : {voting_metrics['ROC-AUC (%)']:.2f}%")

print("\nVoting Classifier complete ✓")


In [ ]:
all_model_names  = list(dhp_results_dict.keys())
all_metric_names = list(list(dhp_results_dict.values())[0].keys())

# ── Table 1: Full comparison (DHP vs GSCV vs RSCV) ─────────
table1_rows = []
for model_name in all_model_names:
    row = {'Model': model_name}
    for method_label, results_source in [
            ('DHP',  dhp_results_dict),
            ('GSCV', gscv_results_dict),
            ('RSCV', rscv_results_dict)]:
        for metric in all_metric_names:
            row[f'{method_label} — {metric}'] = results_source[model_name][metric]
    table1_rows.append(row)

# Add Voting Classifier as last row
voting_row = {'Model': 'Voting Ensemble (Proposed)'}
for method_label in ['DHP', 'GSCV', 'RSCV']:
    for metric in all_metric_names:
        voting_row[f'{method_label} — {metric}'] = voting_metrics[metric]
table1_rows.append(voting_row)

results_table1_df = pd.DataFrame(table1_rows).set_index('Model')

print("\n" + "=" * 70)
print("  TABLE 1 — Model Performance Comparison (DHP vs GSCV vs RSCV)")
print("  All values in % — higher is better")
print("=" * 70)
print(results_table1_df.to_string())

# ── Table 2: k-Fold Cross-Validation results ────────────────
results_table2_df = pd.DataFrame(kfold_results_dict).T
results_table2_df.index.name = 'Model'

print("\n" + "=" * 70)
print("  TABLE 2 — 10-Fold Stratified Cross-Validation Results")
print("  Format: Mean ± Std (all values in %)")
print("=" * 70)
print(results_table2_df.to_string())


In [ ]:
method_names   = ['DHP', 'GSCV', 'RSCV']
results_by_method = {
    'DHP' : dhp_results_dict,
    'GSCV': gscv_results_dict,
    'RSCV': rscv_results_dict,
}

# Include Voting Classifier as a separate entry
display_names    = list(MODEL_DISPLAY_NAMES.values()) + ['Ensemble']
model_keys       = all_model_names + ['Voting Ensemble (Proposed)']

bar_positions    = np.arange(len(display_names))
method_bar_width = 0.22
method_colors    = ['#534AB7', '#1D9E75', '#EF9F27']

fig, ax = plt.subplots(figsize=(16, 7))

for method_idx, (method_label, method_color) in enumerate(
        zip(method_names, method_colors)):

    f1_scores = []
    for model_name in all_model_names:
        f1_scores.append(results_by_method[method_label][model_name]['F1-Score (%)'])
    f1_scores.append(voting_metrics['F1-Score (%)'])  # Voting

    bar_offset = (method_idx - 1) * method_bar_width
    bars = ax.bar(bar_positions + bar_offset,
                  f1_scores,
                  width  = method_bar_width,
                  label  = method_label,
                  color  = method_color,
                  alpha  = 0.88,
                  edgecolor='white',
                  linewidth=0.5)

    for bar, score in zip(bars, f1_scores):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.3,
                f'{score:.1f}',
                ha='center', va='bottom',
                fontsize=7.5, fontweight='bold', rotation=90)

ax.set_xlabel('Classifier', fontsize=12)
ax.set_ylabel('F1-Score (%)', fontsize=12)
ax.set_title('Figure 15 — F1-Score Comparison: DHP vs GSCV vs RSCV\n'
             'Across All 7 Classifiers + Proposed Voting Ensemble',
             fontsize=13, fontweight='bold')
ax.set_xticks(bar_positions)
ax.set_xticklabels(display_names, fontsize=11)
ax.set_ylim(0, 115)
ax.legend(fontsize=11, loc='upper left')
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))

plt.tight_layout()
plt.savefig(RESULTS_PATH + 'fig15_f1_score_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig15_f1_score_comparison.png")


In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))

for method_idx, (method_label, method_color) in enumerate(
        zip(method_names, method_colors)):

    accuracy_scores = []
    for model_name in all_model_names:
        accuracy_scores.append(
            results_by_method[method_label][model_name]['Accuracy (%)'])
    accuracy_scores.append(voting_metrics['Accuracy (%)'])

    bar_offset = (method_idx - 1) * method_bar_width
    bars = ax.bar(bar_positions + bar_offset,
                  accuracy_scores,
                  width     = method_bar_width,
                  label     = method_label,
                  color     = method_color,
                  alpha     = 0.88,
                  edgecolor = 'white',
                  linewidth = 0.5)

    for bar, score in zip(bars, accuracy_scores):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.2,
                f'{score:.1f}',
                ha='center', va='bottom',
                fontsize=7.5, fontweight='bold', rotation=90)

ax.set_xlabel('Classifier', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Figure 16 — Accuracy Comparison: DHP vs GSCV vs RSCV\n'
             'Across All 7 Classifiers + Proposed Voting Ensemble',
             fontsize=13, fontweight='bold')
ax.set_xticks(bar_positions)
ax.set_xticklabels(display_names, fontsize=11)
ax.set_ylim(60, 110)
ax.legend(fontsize=11, loc='lower right')
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))

plt.tight_layout()
plt.savefig(RESULTS_PATH + 'fig16_accuracy_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig16_accuracy_comparison.png")


In [ ]:
roc_line_colors = ['#534AB7', '#1D9E75', '#EF9F27', '#E24B4A',
                   '#0C447C', '#854F0B', '#712B13', '#000000']

fig, ax = plt.subplots(figsize=(10, 8))

# Plot ROC for each GSCV-tuned model
for (model_name, trained_model), line_color in zip(
        gscv_trained_dict.items(), roc_line_colors):

    y_probabilities = trained_model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _     = roc_curve(y_test, y_probabilities)
    auc_score       = roc_auc_score(y_test, y_probabilities)
    short_name      = MODEL_DISPLAY_NAMES[model_name]

    ax.plot(fpr, tpr,
            label     = f'{short_name}  (AUC = {auc_score:.4f})',
            color     = line_color,
            linewidth = 2)

# Plot ROC for Voting Classifier
voting_probs    = voting_classifier.predict_proba(X_test_scaled)[:, 1]
v_fpr, v_tpr, _ = roc_curve(y_test, voting_probs)
v_auc           = roc_auc_score(y_test, voting_probs)
ax.plot(v_fpr, v_tpr,
        label     = f'Ensemble  (AUC = {v_auc:.4f})',
        color     = 'black',
        linewidth = 2.5,
        linestyle = '--')

# Random guess baseline
ax.plot([0, 1], [0, 1],
        color='gray', linestyle=':', linewidth=1.5,
        label='Random Guess (AUC = 0.5000)')

ax.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
ax.set_ylabel('True Positive Rate (Sensitivity / Recall)', fontsize=12)
ax.set_title('Figure 17 — ROC Curves: GSCV-Tuned Models + Proposed Ensemble\n'
             '(Higher AUC = Better Model)',
             fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.grid(linestyle='--', alpha=0.35)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])

plt.tight_layout()
plt.savefig(RESULTS_PATH + 'fig17_roc_curves.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig17_roc_curves.png")


In [ ]:
heatmap_data_dict = {}
for model_name in all_model_names:
    heatmap_data_dict[model_name] = {
        'DHP' : dhp_results_dict [model_name]['Accuracy (%)'],
        'GSCV': gscv_results_dict[model_name]['Accuracy (%)'],
        'RSCV': rscv_results_dict[model_name]['Accuracy (%)'],
    }
heatmap_data_dict['Voting Ensemble'] = {
    'DHP' : voting_metrics['Accuracy (%)'],
    'GSCV': voting_metrics['Accuracy (%)'],
    'RSCV': voting_metrics['Accuracy (%)'],
}

heatmap_df = pd.DataFrame(heatmap_data_dict).T
heatmap_df.index   = list(MODEL_DISPLAY_NAMES.values()) + ['Ensemble']

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(heatmap_df,
            annot=True,
            fmt='.2f',
            cmap='YlGn',
            vmin=heatmap_df.values.min() - 2,
            vmax=100,
            linewidths=0.8,
            linecolor='white',
            annot_kws={'size': 12, 'weight': 'bold'},
            cbar_kws={'label': 'Accuracy (%)', 'shrink': 0.8},
            ax=ax)

ax.set_title('Figure 18 — Accuracy Heatmap (%)\n'
             'All Models × All Optimization Methods',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Optimization Method', fontsize=11)
ax.set_ylabel('Classifier', fontsize=11)
ax.tick_params(axis='x', labelsize=11)
ax.tick_params(axis='y', labelsize=10, rotation=0)

plt.tight_layout()
plt.savefig(RESULTS_PATH + 'fig18_accuracy_heatmap.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig18_accuracy_heatmap.png")


In [ ]:
# Find the best GSCV model by F1-Score
best_gscv_model_name = max(
    gscv_results_dict,
    key=lambda m: gscv_results_dict[m]['F1-Score (%)'])
best_gscv_model      = gscv_trained_dict[best_gscv_model_name]

print(f"Best GSCV model: {best_gscv_model_name}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Figure 19 — Confusion Matrices\n'
             f'Left: Best Individual Model ({best_gscv_model_name} — GSCV)  |  '
             'Right: Proposed Voting Ensemble',
             fontsize=12, fontweight='bold')

for ax, model_obj, model_title in zip(
        axes,
        [best_gscv_model, voting_classifier],
        [f'{best_gscv_model_name}\n(GSCV-tuned)', 'Voting Ensemble\n(Proposed)']):

    y_pred   = model_obj.predict(X_test_scaled)
    cm       = confusion_matrix(y_test, y_pred)
    disp     = ConfusionMatrixDisplay(
        confusion_matrix = cm,
        display_labels   = ['Non-Diabetic\n(0)', 'Diabetic\n(1)'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(model_title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted Label', fontsize=10)
    ax.set_ylabel('True Label',      fontsize=10)

plt.tight_layout()
plt.savefig(RESULTS_PATH + 'fig19_confusion_matrices.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig19_confusion_matrices.png")


In [ ]:
kfold_model_names  = list(kfold_results_dict.keys())
kfold_means        = [kfold_results_dict[m]['Mean Accuracy (%)']
                      for m in kfold_model_names]
kfold_stds         = [kfold_results_dict[m]['Std Accuracy (%)']
                      for m in kfold_model_names]
kfold_short_names  = [MODEL_DISPLAY_NAMES[m] for m in kfold_model_names]

fig, ax = plt.subplots(figsize=(11, 6))

bars = ax.bar(kfold_short_names, kfold_means,
              color=PALETTE[:len(kfold_model_names)],
              edgecolor='white', linewidth=0.8,
              alpha=0.88, width=0.55)

ax.errorbar(kfold_short_names, kfold_means,
            yerr=kfold_stds,
            fmt='none', color='black',
            capsize=6, capthick=1.5, elinewidth=1.5)

for bar, mean, std in zip(bars, kfold_means, kfold_stds):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + std + 0.4,
            f'{mean:.2f}%\n±{std:.2f}%',
            ha='center', va='bottom',
            fontsize=9, fontweight='bold')

ax.set_xlabel('Classifier', fontsize=12)
ax.set_ylabel('Mean Accuracy (%)', fontsize=12)
ax.set_title('Figure 20 — 10-Fold Stratified Cross-Validation Results\n'
             'Mean Accuracy ± Standard Deviation for Each Classifier',
             fontsize=13, fontweight='bold')
ax.set_ylim(0, 115)
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))

plt.tight_layout()
plt.savefig(RESULTS_PATH + 'fig20_kfold_cv_results.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig20_kfold_cv_results.png")


In [ ]:
radar_metrics   = ['Accuracy (%)', 'Precision (%)',
                   'Recall (%)', 'F1-Score (%)', 'ROC-AUC (%)']
radar_labels    = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
num_radar_vars  = len(radar_metrics)
radar_angles    = np.linspace(0, 2 * np.pi, num_radar_vars,
                               endpoint=False).tolist()
radar_angles   += radar_angles[:1]   # close the polygon

# Pick top 4 models by GSCV F1-Score for readability
top4_models = sorted(gscv_results_dict,
                     key=lambda m: gscv_results_dict[m]['F1-Score (%)'],
                     reverse=True)[:4]
top4_models.append('Voting Ensemble (Proposed)')

fig, ax = plt.subplots(figsize=(9, 9),
                       subplot_kw=dict(polar=True))

radar_colors = ['#534AB7', '#1D9E75', '#EF9F27', '#E24B4A', '#000000']

for model_name, radar_color in zip(top4_models, radar_colors):
    if model_name == 'Voting Ensemble (Proposed)':
        metric_values = [voting_metrics[m] for m in radar_metrics]
        display_label = 'Ensemble (Proposed)'
        line_style    = '--'
        line_width    = 2.5
    else:
        metric_values = [gscv_results_dict[model_name][m]
                         for m in radar_metrics]
        display_label = MODEL_DISPLAY_NAMES[model_name]
        line_style    = '-'
        line_width    = 2

    values_closed = metric_values + metric_values[:1]

    ax.plot(radar_angles, values_closed,
            color=radar_color, linewidth=line_width,
            linestyle=line_style, label=display_label)
    ax.fill(radar_angles, values_closed,
            color=radar_color, alpha=0.08)

ax.set_xticks(radar_angles[:-1])
ax.set_xticklabels(radar_labels, fontsize=11)
ax.set_ylim(60, 100)
ax.set_yticks([65, 70, 75, 80, 85, 90, 95, 100])
ax.set_yticklabels(['65%','70%','75%','80%','85%','90%','95%','100%'],
                   fontsize=8)
ax.set_title('Figure 21 — Radar Chart: Multi-Metric Comparison\n'
             'Top 4 Models (GSCV) + Proposed Ensemble',
             fontsize=12, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_PATH + 'fig21_radar_chart.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig21_radar_chart.png")


In [ ]:
# Table 1 — Full DHP / GSCV / RSCV comparison
results_table1_df.to_csv(
    RESULTS_PATH + 'table1_dhp_gscv_rscv_comparison.csv')
print("Saved: table1_dhp_gscv_rscv_comparison.csv")

# Table 2 — k-Fold Cross-Validation results
results_table2_df.to_csv(
    RESULTS_PATH + 'table2_kfold_cv_results.csv')
print("Saved: table2_kfold_cv_results.csv")

# Table 3 — Best params from GSCV
gscv_params_df = pd.DataFrame(gscv_best_params_dict).T
gscv_params_df.index.name = 'Model'
gscv_params_df.to_csv(RESULTS_PATH + 'table3_gscv_best_params.csv')
print("Saved: table3_gscv_best_params.csv")

# Table 4 — Best params from RSCV
rscv_params_df = pd.DataFrame(rscv_best_params_dict).T
rscv_params_df.index.name = 'Model'
rscv_params_df.to_csv(RESULTS_PATH + 'table4_rscv_best_params.csv')
print("Saved: table4_rscv_best_params.csv")

# Table 5 — Feature importance scores
feature_scores.to_csv(
    RESULTS_PATH + 'table5_feature_importance_selectkbest.csv',
    index=False)
rf_importance_df.to_csv(
    RESULTS_PATH + 'table6_feature_importance_randomforest.csv',
    index=False)
print("Saved: table5 and table6 — feature importance scores")


In [ ]:
all_metric_names = list(list(dhp_results_dict.values())[0].keys())

for metric_name in all_metric_names:
    best_model_name = max(
        gscv_results_dict,
        key=lambda m: gscv_results_dict[m][metric_name])
    best_score = gscv_results_dict[best_model_name][metric_name]
    print(f"  {metric_name:18s}: {best_model_name:22s} → {best_score:.2f}%")

print(f"\n  Voting Ensemble (Proposed):")
for metric_name, metric_value in voting_metrics.items():
    print(f"    {metric_name:18s}: {metric_value:.2f}%")

print("\n" + "=" * 65)
print("  ALL FILES SAVED TO GOOGLE DRIVE:")
print(f"  {RESULTS_PATH}")
print("=" * 65)
print("""
  PNG FIGURES:
    fig01  — Histograms (numerical features)
    fig02  — Box plots (numerical features)
    fig03  — KDE density plots
    fig04  — Target class distribution
    fig05  — Categorical distributions
    fig06  — Binary feature distributions
    fig07  — Numerical features vs target
    fig08  — Correlation heatmap
    fig09  — Scatter plots (key pairs)
    fig10  — Scatter plot matrix (pair plot)
    fig11  — Categorical features vs target
    fig12  — SMOTE before vs after
    fig13  — Feature importance (SelectKBest)
    fig14  — Feature importance (Random Forest)
    fig15  — F1-Score comparison (all methods)
    fig16  — Accuracy comparison (all methods)
    fig17  — ROC curves (GSCV + Ensemble)
    fig18  — Accuracy heatmap
    fig19  — Confusion matrices
    fig20  — k-Fold CV mean ± std
    fig21  — Radar chart (multi-metric)

  CSV TABLES (open in Excel → format for paper):
    table1 — DHP / GSCV / RSCV full comparison
    table2 — 10-Fold CV mean ± std results
    table3 — GSCV best hyperparameters
    table4 — RSCV best hyperparameters
    table5 — SelectKBest feature scores
    table6 — Random Forest feature importance
""")
print("  DONE. Good luck with ICDAI-2026!")
print("=" * 65)